# Obesity Trends in US Children and Adolescents (2-19)

CDC/NCHS NHANES, ten survey cycles from 1988-1994 to 2015-2018.

In [ ]:
%matplotlib inline
import sys

sys.path.insert(0, "../src")

import pandas as pd

from obesity.load import available_groups, series_for, tidy
from obesity.model import fit_all_groups, fit_trend
from obesity.plots import (
    plot_groups,
    plot_race_small_multiples,
    plot_trend,
    save_all,
)

pd.set_option("display.width", 200)

## 1. Load

In [ ]:
frame = tidy()
print(frame.shape)
frame.head()

In [ ]:
frame[["cycle", "year_window", "year_mid"]].drop_duplicates().sort_values("cycle")

## 2. What can be modelled

Groups are only comparable at the all-ages (age_num == 0) level and need at least two cycles to fit a line at all.

In [ ]:
available_groups(frame)

## 3. The national trend

Each group gets a plain least-squares line against `year_mid`, the survey
window midpoint (cycles are unevenly spaced, so the slope is percent per year,
not percent per cycle).

In [ ]:
total = fit_trend(frame, "total", "2-19 years")

print(f"slope: {total.slope_per_decade:+.2f} pp/decade   cycles: {total.n_obs}")

series_for(frame, "total", "2-19 years")[["year_window", "estimate", "se"]]

In [ ]:
plot_trend(frame, "total", "2-19 years");

Note the shape of the residuals: prevalence climbed steeply to about 2002 and
has been much flatter since. 
- read the slope as a rough summary
of the whole period rather than the current rate of change.

## 4. Every group's slope

In [ ]:
models = fit_all_groups(frame)

slopes = pd.DataFrame(
    [
        {
            "group_kind": kind,
            "group_label": label,
            "slope_per_decade": round(model.slope_per_decade, 2),
            "n": model.n_obs,
        }
        for (kind, label), model in models.items()
    ]
).sort_values("group_kind", ascending=False)

slopes.reset_index(drop=True)

Every group that can be fitted is rising except the highest income bracket
(400%+ of the poverty level), whose slope is essentially flat.

In [ ]:
plot_groups(frame, "sex", title="Obesity prevalence by sex, ages 2-19");

In [ ]:
plot_groups(
    frame, "poverty", title="Obesity prevalence by percent of poverty level, ages 2-19"
);

In [ ]:
plot_groups(frame, "age", title="Obesity prevalence by age band");

In [ ]:
plot_race_small_multiples(frame);

In [ ]:
save_all(frame)

This project is descriptive: obesity prevalence among US 2-19 year-olds roughly doubled between 1988-1994 and 2015-2018, the increase is steepest among adolescents and children in low-to-middle income brackets, and it is flat only in the highest income bracket.